In [1]:
import pandas as pd
import numpy as np

In [2]:
retail= pd.read_csv('data.nosync/online_retail_II.csv')
retail.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [123]:
def cleaning(df):
    df_clean = df.copy()
    print(f"Rows before cleaning: {len(df)}")
    # 1. Dropping missing customerid
    df_clean = df_clean.dropna(subset=["Customer ID"])
    # 2. Standardizing invoice dates to datetime format
    df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'],errors="coerce")
    df_clean=df_clean.dropna(subset=['InvoiceDate']) # Removing NAT datetime after conversion 
    # 3. Filling missing value in description columns
    df_clean["Description"] = df_clean["Description"].fillna("unknown")
    # 4. Removing invalid rows
    df_clean=df_clean.loc[(df_clean["Quantity"]>0) & (df_clean["Price"]>0),:].copy()
    # 5. Feature Engineering
    df_clean["total_price"] = df_clean["Quantity"] * df_clean["Price"]
    df_clean["Invoice_year"] = df_clean['InvoiceDate'].dt.year
    df_clean["Invoice_month"] = df_clean['InvoiceDate'].dt.month
    # 5. Standarizing columns name to snake case
    df_clean.columns = df_clean.columns.str.strip().str.lower().str.replace(" ","_")
    df_clean.rename(columns={"Price":"Unit_price"},inplace=True)
    print(f"Rows After cleaning: {len(df_clean)}\nRows dropped: {(len(df)-len(df_clean))*100/len(df):.2f}%")
    return df_clean.reset_index(drop=True)
retail_clean = cleaning(retail)

Rows before cleaning: 1067371
Rows After cleaning: 805549
Rows dropped: 24.53%


In [104]:
##### print("--- Structure ---")
print(retail_clean.info())
print("--- Missing Value ---")
print(retail_clean.isna().sum())
print("--- Overview ---")
print(retail_clean.describe())

<class 'pandas.DataFrame'>
RangeIndex: 805549 entries, 0 to 805548
Data columns (total 11 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   invoice        805549 non-null  str           
 1   stockcode      805549 non-null  str           
 2   description    805549 non-null  str           
 3   quantity       805549 non-null  int64         
 4   invoicedate    805549 non-null  datetime64[us]
 5   price          805549 non-null  float64       
 6   customer_id    805549 non-null  float64       
 7   country        805549 non-null  str           
 8   total_price    805549 non-null  float64       
 9   invoice_year   805549 non-null  int32         
 10  invoice_month  805549 non-null  int32         
dtypes: datetime64[us](1), float64(3), int32(2), int64(1), str(4)
memory usage: 101.1 MB
None
--- Missing Value ---
invoice          0
stockcode        0
description      0
quantity         0
invoicedate      0
price        

In [126]:
retail_clean['invoicedate'].dt.date.floor()

AttributeError: 'Series' object has no attribute 'floor'